#  link predictor

In [60]:
#Importing necessary foundations and basic setup

!pip install torch_geometric
import itertools
import copy
import random
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import roc_auc_score,average_precision_score

dataset = Planetoid(root="data/planetoid",name="Cora",transform = NormalizeFeatures())
data = dataset[0]

print(data)
print("Nodes: ", data.num_nodes)
print("Directed edge entries: ", data.edge_index.size(1) )
print("Feature dimensions:", data.num_node_features)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Nodes:  2708
Directed edge entries:  10556
Feature dimensions: 1433


In [32]:
# Extracting unique positive links
edges = data.edge_index.t()

#sorting them to make the undirectl links duplicate
positive_edges = torch.sort(edges,dim=1).values
#removing the duplicated edges
positive_edges = torch.unique(positive_edges,dim = 0)
#removing self loop if exists
positive_edges = positive_edges[
    positive_edges[:,0] != positive_edges[:,1]
]

print(f"Unique positive links: {positive_edges.size(0)}")
print("Number of nodes: ", data.num_nodes)

Unique positive links: 5278
Number of nodes:  2708


Splitting positive link to train, val , and test masks and constructing the edge index

In [33]:
seed = 0
random.seed(seed)
torch.manual_seed(seed)

#shuffling
n = positive_edges.size(0)
perm = torch.randperm(n)
positive_edges = positive_edges[perm]

n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_pos = positive_edges[:n_train]
val_pos = positive_edges[n_train:n_train+n_val]
test_pos = positive_edges[n_train+n_val:]

print("Train positives:", len(train_pos))
print("Validation positives:", len(val_pos))
print("Test positives:", len(test_pos))


train_edge_index = torch.cat(
    [train_pos.t(),train_pos.flip(1).t()],
    dim=1
)

print(f"Training message pas graph: {train_edge_index.shape}")
print(f"Training positive link: {train_pos.shape}")

Train positives: 4222
Validation positives: 527
Test positives: 529
Training message pas graph: torch.Size([2, 8444])
Training positive link: torch.Size([4222, 2])


Adding negative test using iteration

In [34]:
rng = random.Random(seed)
num_nodes = data.num_nodes
all_positive = {
    (int(u), int(v))
    for u, v in positive_edges.tolist()
}
num_negative = len(train_pos) + len(test_pos) + len(val_pos)
negative_set = set()
while len(negative_set) < num_negative:
  u = rng.randrange(num_nodes)
  v = rng.randrange(num_nodes)
  if u == v:
    continue

  pair = (min(u,v), max(u,v))

  if pair not in negative_set and pair not in all_positive:
    negative_set.add(pair)


negative_edges = torch.tensor(
    sorted(negative_set),
    dtype=torch.long
)
n_train_neg = len(train_pos)
n_val_neg = len(val_pos)


train_neg = negative_edges[:n_train_neg]
val_neg = negative_edges[n_train_neg:n_train_neg+n_val_neg]
test_neg = negative_edges[n_train_neg + n_val_neg:]

print("Train negitives:", len(train_neg))
print("Validation negitives:", len(val_neg))
print("Test negitives:", len(test_neg))

Train negitives: 4222
Validation negitives: 527
Test negitives: 529


automated lekage check

In [35]:
def edge_set(edge_index):
  edges = edge_index.t().tolist()
  return {(u,v) for u,v in edges}

message_edges = edge_set(train_edge_index)

def assert_absent(held_out_edges,graph_edges):
  for u,v in held_out_edges.tolist():
    assert (u,v) not in graph_edges, (u,v)
    assert (v, u) not in graph_edges, (v,u)

assert not (negative_set & all_positive)

assert_absent(val_pos,message_edges)
assert_absent(test_pos,message_edges)

train_set = set(map(tuple,train_pos.tolist()))
val_set = set(map(tuple,val_pos.tolist()))
test_set = set(map(tuple,test_pos.tolist()))

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

train_neg_set = set(map(tuple,train_neg.tolist()))
val_neg_set = set(map(tuple,val_neg.tolist()))
test_neg_set = set(map(tuple,test_neg.tolist()))

assert train_neg_set.isdisjoint(val_neg_set)
assert train_neg_set.isdisjoint(test_neg_set)
assert val_neg_set.isdisjoint(test_neg_set)

assert len(train_neg) == len(train_pos)
assert len(val_neg) == len(val_pos)
assert len(test_neg) == len(test_pos)
print("Train:", len(train_pos), "positive,", len(train_neg), "negative")
print("Val:  ", len(val_pos), "positive,", len(val_neg), "negative")
print("Test: ", len(test_pos), "positive,", len(test_neg), "negative")


Train: 4222 positive, 4222 negative
Val:   527 positive, 527 negative
Test:  529 positive, 529 negative


GCN layer

In [36]:
class GCNEncoder(torch.nn.Module):
  def __init__(self,in_dim,hidden,out_dim):
    super().__init__()
    self.conv1 = GCNConv(in_dim,hidden)
    self.conv2 = GCNConv(hidden,out_dim)

  def forward(self,x,edge_index):
    h = F.relu(self.conv1(x,edge_index))
    return self.conv2(h,edge_index)


encoder = GCNEncoder(
  data.num_node_features,
  64,
  32
)
x = data.x
encoder.eval()
with torch.no_grad():
  z = encoder(x,train_edge_index)
print(f"Input features: {x.shape}")
print(f"Node embeddings:  {z.shape}")
print(f"Embedding for paper0: {z[0]}")

Input features: torch.Size([2708, 1433])
Node embeddings:  torch.Size([2708, 32])
Embedding for paper0: tensor([-0.0067,  0.0034, -0.0051,  0.0101,  0.0023,  0.0044,  0.0148, -0.0053,
         0.0023, -0.0030, -0.0030, -0.0150, -0.0028,  0.0070,  0.0083,  0.0020,
         0.0007, -0.0079,  0.0022,  0.0014, -0.0056,  0.0043, -0.0095, -0.0013,
        -0.0012,  0.0060, -0.0045,  0.0019, -0.0004, -0.0093, -0.0007, -0.0009])


Constructing dot product

In [52]:
def dot_product_decoder(Z,edge_pairs):
  u = edge_pairs[:,0]
  v = edge_pairs[:,1]

  z_u = Z[u]
  z_v = Z[v]


  return (z_u * z_v).sum(dim=1)

MLP decoder


In [38]:
class symetricMLPDecoder(torch.nn.Module):
  def __init__(self,embedding_dim,hidden_dim=64,dropout=0.5):
    super().__init__()
    self.l1 = torch.nn.Linear(2 * embedding_dim,hidden_dim)
    self.l2 = torch.nn.Linear(hidden_dim,1)
    self.dropout = dropout
  def forward(self,z,edge_pairs):
    u = edge_pairs[:,0]
    v = edge_pairs[:,1]

    z_u = z[u]
    z_v = z[v]

    difference = torch.abs(z_u - z_v)
    product = z_u * z_v
    pair_features = torch.cat(
        [difference,product], dim=1
    )
    x = F.dropout(pair_features,p=self.dropout, training = self.training)
    x = F.relu(self.l1(x))
    x = F.dropout(x,p=self.dropout, training = self.training)
    return self.l2(x).squeeze(-1)




In [53]:
mlp_decoder = symetricMLPDecoder(32,6)
pairs = torch.tensor([[0,1],[2,3],[10,20]],dtype=torch.long)
reversed_pairs = pairs.flip(1)
encoder.eval()
mlp_decoder.eval()

with torch.no_grad():
  z = encoder(x,train_edge_index)
  dot_forward = dot_product_decoder(z,pairs)
  dot_reverse = dot_product_decoder(z,reversed_pairs)

  mlp_forward = mlp_decoder(z,pairs)
  mlp_reverse = mlp_decoder(z,reversed_pairs)

assert torch.allclose(dot_forward,dot_reverse,atol=1e-6)
assert torch.allclose(mlp_forward,mlp_reverse,atol=1e-6)


Combining negative and positive examples


In [54]:
train_pairs = torch.cat([train_pos,train_neg],dim=0)
train_labels = torch.cat([torch.ones(len(train_pos)),torch.zeros(len(train_neg))])

val_pairs = torch.cat([val_pos,val_neg],dim=0)
val_labels = torch.cat([torch.ones(len(val_pos)),torch.zeros(len(val_neg))])

test_pairs = torch.cat([test_pos,test_neg],dim=0)
test_labels = torch.cat([torch.ones(len(test_pos)),torch.zeros(len(test_neg))])

Creating trianing function for both decoder

In [67]:
def trian_link_predicator(encoder,decoder,epochs=100,lr=0.01):
  params = list(encoder.parameters())
  if decoder != "dot":
    params += list(decoder.parameters())
  loss_fn = torch.nn.BCEWithLogitsLoss()
  optimizer = torch.optim.Adam(params,lr=lr)
  best_encoder_state, best_decoder_state,best_value_acc = None, None, -float('inf')
  for epoch in range(epochs):
    encoder.train()
    if decoder != "dot":
      decoder.train()

    optimizer.zero_grad()
    z = encoder(x,train_edge_index)
    if decoder == "dot":
      train_scores = dot_product_decoder(z,train_pairs)
    else:
      train_scores = decoder(z,train_pairs)
    loss = loss_fn(train_scores,train_labels)
    loss.backward()
    optimizer.step()

    encoder.eval()
    if decoder != "dot":
      decoder.eval()

    with torch.no_grad():
      z_val = encoder(x,train_edge_index)
      if decoder == "dot":
        val_scores = dot_product_decoder(z,val_pairs)
      else:
        val_scores = decoder(z,val_pairs)

      #reciving operating characterstics- area under curve
      val_auc = roc_auc_score(val_labels.numpy(),val_scores.numpy())
      if val_auc > best_value_acc:
        best_value_acc = val_auc
        best_encoder_state = copy.deepcopy(encoder.state_dict())
        if decoder != "dot":
          best_decoder_state = copy.deepcopy(decoder.state_dict())

      if (epoch + 1 ) % 20 == 0:
        print(
        f"Epoch {epoch + 1:3d} | "
        f"Loss: {loss.item():.4f} | "
        f"Val ROC-AUC: {val_auc:.4f}"
        )
  encoder.load_state_dict(best_encoder_state)
  if decoder != "dot":
    decoder.load_state_dict(best_decoder_state)

  return encoder,decoder,best_value_acc


In [76]:
#rinanig both models
dot_encoder = GCNEncoder(data.num_node_features,64,32)
dot_encoder, _ , dot_best_ap = trian_link_predicator(
    dot_encoder,"dot"
)

print("MLP result")
#rinanig 2nd models
mlp_encoder = GCNEncoder(data.num_node_features,64,32)
mlp_decoder = symetricMLPDecoder(embedding_dim=32,hidden_dim=64)

mlp_encoder, mlp_decoder, mlp_best_ap = trian_link_predicator(
    mlp_encoder,mlp_decoder
)
print("Best validation AP — dot product:", dot_best_ap)
print("Best validation AP — MLP:", mlp_best_ap)

Epoch  20 | Loss: 0.5814 | Val ROC-AUC: 0.7261
Epoch  40 | Loss: 0.4449 | Val ROC-AUC: 0.8150
Epoch  60 | Loss: 0.3514 | Val ROC-AUC: 0.8300
Epoch  80 | Loss: 0.2692 | Val ROC-AUC: 0.8185
Epoch 100 | Loss: 0.1941 | Val ROC-AUC: 0.7927
MLP result
Epoch  20 | Loss: 0.3555 | Val ROC-AUC: 0.8429
Epoch  40 | Loss: 0.1490 | Val ROC-AUC: 0.9153
Epoch  60 | Loss: 0.0839 | Val ROC-AUC: 0.9054
Epoch  80 | Loss: 0.0650 | Val ROC-AUC: 0.9041
Epoch 100 | Loss: 0.0527 | Val ROC-AUC: 0.9018
Best validation AP — dot product: 0.8381335762559906
Best validation AP — MLP: 0.9152699213981975


In [70]:
#helper function
def score_pairs(encoder,decoder,x,edge_index,pairs):
  z = encoder(x,edge_index)
  if decoder == "dot":
      return dot_product_decoder(z, pairs)

  return decoder(z, pairs)

In [75]:
def evaluate_link_model(encoder,decoder,test_pairs,test_labels,x,train_edge_index):
  encoder.eval()
  if decoder!="dot":
    decoder.eval()
  with torch.no_grad():
    logits = score_pairs(encoder,decoder,x,train_edge_index,test_pairs)
    probs = torch.sigmoid(logits).numpy()
  labels = test_labels.numpy()
  return {
        "ROC-AUC": roc_auc_score(labels, probs),
        "Average Precision": average_precision_score(labels, probs)
    }
dot_prodcut_decoder_result = evaluate_link_model(
    dot_encoder, "dot",
    test_pairs, test_labels, x, train_edge_index
)
mlp_decoder_result = evaluate_link_model(
    mlp_encoder, mlp_decoder,
    test_pairs, test_labels, x, train_edge_index
)

print(f"dot product metrics: {dot_prodcut_decoder_result}\n mlp decoder metric: {mlp_decoder_result}")

dot product metrics: {'ROC-AUC': np.float64(0.8224992049056428), 'Average Precision': np.float64(0.8277583088530265)}
 mlp decoder metric: {'ROC-AUC': np.float64(0.9102972759531306), 'Average Precision': np.float64(0.9105880891231087)}


# Conclusion
I can conclude that mlp decoding beats dot product metrics sinc eit is not subjected to simple calculation but also trained in node embedding,  while splitting the data in to separate mask , and preventing data lekage. Link prediction is useful in recommendation social networks, and citation networks. where it can help identify potential connections that have not yet been observed